        # 00 - Environment check

        Run this notebook **before** the workshop to make sure your environment
        can reach the cloud and stream data. It takes well under a minute.

        If every cell runs without an error, you are ready to go.
        

## 1. Packages and versions

In [ ]:
        import sys

        import dask
        import matplotlib
        import numpy as np
        import odc.stac
        import pandas as pd
        import planetary_computer
        import pystac_client
        import rasterio
        import scipy
        import xarray as xr

        import eo_utils as eo

        print("python      ", sys.version.split()[0])
        print("odc-stac    ", odc.stac.__version__)
        print("xarray      ", xr.__version__)
        print("dask        ", dask.__version__)
        print("rasterio    ", rasterio.__version__)
        print("pystac-client", pystac_client.__version__)
        print("scipy       ", scipy.__version__)
        print("pandas      ", pd.__version__)
        print("matplotlib  ", matplotlib.__version__)
        print()
        print("eo_utils loaded from:", eo.__file__)
        

## 2. Cloud-friendly defaults

In [ ]:
        eo.apply_cloud_defaults()
        print("GDAL retries + limited Dask connections configured.")
        

## 3. Connect to the STAC catalogues

In [ ]:
        pc = eo.open_planetary_computer()
        thuenen = eo.open_thuenen()

        print("Planetary Computer :", eo.PLANETARY_COMPUTER_URL)
        print("Thünen Institute   :", eo.THUENEN_STAC_URL)
        print()
        print("Thünen collections :", [c.id for c in thuenen.get_collections()])
        

## 4. Stream a tiny slice of Landsat (no download)

In [ ]:
        bbox = [8.98, 53.28, 9.08, 53.32]  # ~7 x 4 km, Lower Saxony

        items = eo.search_signed_landsat(pc, bbox, "2020-06-01/2020-06-30", cloud_cover_lt=20)
        print(f"{len(items)} Landsat scenes found")

        ds = eo.load_landsat(items, bbox, chunks=(-1, 256, 256))
        ndvi = eo.compute_ndvi(ds, clear_mask=eo.landsat_clear_mask(ds))

        with eo.compute_with_progress():
            result = ndvi.compute()

        print("NDVI stack shape:", result.shape)
        print("Mean NDVI        :", round(float(np.nanmean(result)), 3))
        

## 5. Stream a tiny slice of the Thünen crop map and soil composite

In [ ]:
        crops = eo.search_crop_types(thuenen, bbox, "2020-01-01/2020-12-31")
        soil = eo.search_soil_composite(thuenen, bbox)
        print("crop-map items :", len(crops))
        print("soil items     :", len(soil))

        crop_ds = eo.load_crop_types(crops, bbox=bbox, resolution=100, chunks=(-1, 128, 128))
        with eo.compute_with_progress():
            crop_ds = crop_ds.compute()
        print("crop_type range:", float(crop_ds["crop_type"].min()), "-", float(crop_ds["crop_type"].max()))
        

        ## All good?

        If the cells above ran, your environment is workshop-ready.

        **Troubleshooting**

        * `ModuleNotFoundError` -> you are using the wrong kernel. Select the
          `cloud-native-eo` environment (Kernel -> Change Kernel).
        * HTTP 429 / 503 -> you are being rate-limited. Re-run the cell; the
          GDAL retry settings from `eo.apply_cloud_defaults()` will back off.
        * No internet / corporate proxy -> STAC and cloud streaming need outbound
          HTTPS access to `planetarycomputer.microsoft.com` and
          `eodata.thuenen.de`.
        